## Limpeza e importação dos dados para o Neo4j

In [1]:
import pandas as pd
from neo4j import GraphDatabase
import os
from dotenv import load_dotenv

In [2]:
df_artigos = pd.read_csv('data/professores_artigos.csv', low_memory=False)
df_artigos.dropna()
df_artigos

,id,titulo,tipo,anoPublicado,criador
0,9900625695933750,A definir,<http://purl.org/ontology/bibo/Thesis>,2022,Laís dos Santos Gonçalves
1,9900625695933750,A definir,<http://purl.org/ontology/bibo/Thesis>,2022,MARCOS ROGOZINSKI
2,9900625695933750,A definir,<http://purl.org/ontology/bibo/Thesis>,2021,GUILHERME FERREIRA GUSMÃO
3,9900625695933750,A definir,<http://purl.org/ontology/bibo/Thesis>,2024,BRANDO LUIS MARTÍNEZ CHÁVEZ
4,9900625695933750,A definir,<http://purl.org/ontology/bibo/Thesis>,2024,BRUNO OLIVEIRA DE MELO
...,...,...,...,...,...
261296,9542120311882103,The Subjective Perception of Social Objects: A...,<http://purl.org/ontology/bibo/Article>,2015,Wachelke
261297,9542120311882103,Atitudes de futuros motoristas perante a Políc...,<http://purl.org/ontology/bibo/Article>,2004,Wachelke
261298,9542120311882103,Pollution-Aware Walking in 16 Countries: An Ap...,<http://purl.org/ontology/bibo/Article>,2021,"YUAN, QUAN"
261299,9736083270809253,Visita ao Jardim Botânico do Rio de Janeiro: a...,<http://purl.org/ontology/bibo/Article>,2010,Zanini


In [3]:
df_artigos = df_artigos[pd.to_numeric(df_artigos['anoPublicado'], errors='coerce').notnull()]
df_artigos.loc[:, 'anoPublicado'] = df_artigos.loc[:, 'anoPublicado'].str[-4:]
df_artigos['tipo'] = df_artigos['tipo'].str.extract(r'bibo/(\w+)')
df_artigos = df_artigos.rename(columns={'id':'lattes'})
df_artigos = df_artigos.drop('criador', axis=1)
df_artigos

C:\Users\luiza\AppData\Local\Temp\ipykernel_18084\3703105215.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_artigos['tipo'] = df_artigos['tipo'].str.extract(r'bibo/(\w+)')


,lattes,titulo,tipo,anoPublicado
0,9900625695933750,A definir,Thesis,2022
1,9900625695933750,A definir,Thesis,2022
2,9900625695933750,A definir,Thesis,2021
3,9900625695933750,A definir,Thesis,2024
4,9900625695933750,A definir,Thesis,2024
...,...,...,...,...
261296,9542120311882103,The Subjective Perception of Social Objects: A...,Article,2015
261297,9542120311882103,Atitudes de futuros motoristas perante a Políc...,Article,2004
261298,9542120311882103,Pollution-Aware Walking in 16 Countries: An Ap...,Article,2021
261299,9736083270809253,Visita ao Jardim Botânico do Rio de Janeiro: a...,Article,2010


In [4]:
df_artigos[df_artigos['titulo'].isna()]
df_artigos = df_artigos.replace({'titulo': {None: ''}})
df_artigos[df_artigos['titulo'].isna()]

,lattes,titulo,tipo,anoPublicado


In [5]:
df_deptos = pd.read_csv('data/dados_todos_deptos.csv')
df_deptos.dropna()
df_deptos

,nome,lattes,deptos
0,Vítor Hugo dos Santos Gomes Maia,1107478160614,Biologia
1,Monica Feijo Naccache,10848295031388,Engenharia Mecânica
2,Carlos de Lamare Bastian Pinto,13630384437068,Administração de Empresas
3,Leticia da Costa Paes,19354508631210,Direito
4,Jorge Lucas Ferreira,21400037764938,Ciclo Profissional das Engenharias
...,...,...,...
1087,Eduardo Zilberman,9936389852541404,Economia
1088,Sheila de Barcellos Maia,9944052203924339,Administração de Empresas
1089,Julia Bloomfield Gama Zardo,9959521221278547,Administração de Empresas
1090,Alfonso Garcia Rubio,9995659363482583,Vice-Reitoria para Assuntos Acadêmicos


Para facilitar a importação das produções, filtrei apenas as produzidas pelos professores presentes no dataset acima

In [6]:
dados_juntos = pd.merge(df_deptos, df_artigos, on='lattes', how='inner')
dados_juntos

,nome,lattes,deptos,titulo,tipo,anoPublicado
0,Vítor Hugo dos Santos Gomes Maia,1107478160614,Biologia,"Hijmania, a replacement name for Maria (Moraceae)",Article,2016
1,Vítor Hugo dos Santos Gomes Maia,1107478160614,Biologia,"Maria, a New Genus of Moraceae",Article,2013
2,Vítor Hugo dos Santos Gomes Maia,1107478160614,Biologia,A new subfamily classification of the Legumino...,Article,2017
3,Vítor Hugo dos Santos Gomes Maia,1107478160614,Biologia,"Maria, a New Genus of Moraceae",Article,2013
4,Vítor Hugo dos Santos Gomes Maia,1107478160614,Biologia,"Hijmania, a replacement name for Maria (Moraceae)",Article,2016
...,...,...,...,...,...,...
259703,Paulo Eduardo Ramos de Araujo Penna,9997595944857341,Direito,Oferta pública de aquisição de controle de com...,Chapter,2017
259704,Paulo Eduardo Ramos de Araujo Penna,9997595944857341,Direito,Alienação de controle de companhia aberta,Chapter,2015
259705,Paulo Eduardo Ramos de Araujo Penna,9997595944857341,Direito,Brazil - Investment Adviser Regulation,Chapter,2012
259706,Paulo Eduardo Ramos de Araujo Penna,9997595944857341,Direito,Alienação de controle de companhia aberta,Book,2012


In [7]:
dados_juntos = dados_juntos.drop(['deptos'], axis=1)
dados_juntos = dados_juntos.drop_duplicates()

In [8]:
dados_juntos['anoPublicado'] = pd.to_numeric(dados_juntos['anoPublicado'], errors='coerce')
dados_juntos

,nome,lattes,titulo,tipo,anoPublicado
0,Vítor Hugo dos Santos Gomes Maia,1107478160614,"Hijmania, a replacement name for Maria (Moraceae)",Article,2016
1,Vítor Hugo dos Santos Gomes Maia,1107478160614,"Maria, a New Genus of Moraceae",Article,2013
2,Vítor Hugo dos Santos Gomes Maia,1107478160614,A new subfamily classification of the Legumino...,Article,2017
5,Vítor Hugo dos Santos Gomes Maia,1107478160614,Estudo da Genética Populacional de Grazielanth...,Article,2014
6,Vítor Hugo dos Santos Gomes Maia,1107478160614,Estudo da Diversidade Genética Populacional de...,Article,2015
...,...,...,...,...,...
259703,Paulo Eduardo Ramos de Araujo Penna,9997595944857341,Oferta pública de aquisição de controle de com...,Chapter,2017
259704,Paulo Eduardo Ramos de Araujo Penna,9997595944857341,Alienação de controle de companhia aberta,Chapter,2015
259705,Paulo Eduardo Ramos de Araujo Penna,9997595944857341,Brazil - Investment Adviser Regulation,Chapter,2012
259706,Paulo Eduardo Ramos de Araujo Penna,9997595944857341,Alienação de controle de companhia aberta,Book,2012


### Importação dos dados no Neo4J

In [9]:
prof_list = list()
edges_list = list()

for idx,row in df_deptos.iterrows():
    lattes = row.get('lattes')
    nome = row.get('nome')
    deptos = row.get('deptos')
    edges_list.append({
        "lattes":lattes,
        "deptos": deptos,
    })
    prof_list.append({
        "lattes":lattes,
        "nome":nome,
    })

In [10]:
prod_list = list()

df_artigos = pd.read_csv('data/professores_artigos.csv', low_memory = False)
df_artigos.dropna()

for idx, row in dados_juntos.iterrows():
    lattes_criador = row.get('lattes')
    titulo = row.get('titulo')
    ano = row.get('anoPublicado')
    tipo = row.get('tipo')
    nome_autor = row.get('nome')

    prod_list.append({
        "lattes_criador": lattes_criador,
        "titulo": titulo,
        "ano": ano,
        "tipo": tipo,
        "autor": nome_autor
    })

In [11]:
len(prod_list)

90906

In [12]:
deptos_list = set(df_deptos['deptos'].to_list())

Armazenando producoes para serem usadas em ```embeddings.ipynb```

In [13]:
%store prod_list

Stored 'prod_list' (list)


In [14]:
def init_driver(uri, username, password):
    '''Função que conecta ao banco do Neo4J e verifica se a conexão foi bem sucedida.'''
    driver = GraphDatabase().driver(str(uri), auth=(username, password))
    driver.verify_connectivity()
    return driver

In [15]:
load_dotenv()
NEO4J_USERNAME =  os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD =  os.getenv('NEO4J_PASSWORD')
NEO4J_CONNECTION_URL = os.getenv('NEO4J_URI')

In [16]:
driver = init_driver(NEO4J_CONNECTION_URL, NEO4J_USERNAME, NEO4J_PASSWORD)

In [17]:
def esvazia_banco(driver) -> None:
    """Metodo que esvazia todo o banco Neo4j atraves do driver."""
    query = """MATCH (n) DETACH DELETE n"""
    with driver.session() as session:
        try:
            result = session.run(query)
            print("Banco esvaziado.")
        except Exception as e:
            print(f"Erro: {e}")
        finally:
            session.close()

In [18]:
def insert_dpt_nodes(driver, dpt_list, verbose=False) -> None:
    '''Insere nós dos departamentos no banco Neo4j através do driver.'''
    query = """
    UNWIND $departamentos AS nome_dpt
    MERGE (n:Departamento {nome: nome_dpt})
    RETURN n
    """
    with driver.session() as session:
        try:
            print("Inserindo departamentos...")
            result = session.run(query, departamentos=list(dpt_list))
            if verbose:
                for record in result:
                    print(record)
        except Exception as e:
            print(f"Erro: {e}")
        finally:
            session.close()
            print("Departamentos inseridos!")

In [19]:
def insert_prof_nodes(driver, prof_list, verbose=False):
    '''Insere nós dos professores no banco Neo4j através do driver.'''
    query = """
    UNWIND $professores AS prof
    MERGE (n:Professor {nome: prof.nome, lattes: prof.lattes})
    RETURN n
    """
    with driver.session() as session:
        try:
            print("Inserindo professores...")
            result = session.run(query, professores=prof_list)
            if verbose:
                for record in result:
                    print(record)
        except Exception as e:
            print(f"Erro: {e}")
        finally:
            session.close()
            print("Professores inseridos!")

In [20]:
def insert_rel_prof_dept(driver, edges_list, verbose=False):
    '''Cria a relação entre professores e departamentos no banco do Neo4J através do driver.'''
    query = """
    UNWIND $relacoes AS rel
    MATCH (p:Professor {lattes: rel.lattes})
    MATCH (d:Departamento {nome: rel.deptos})
    MERGE (p)-[:PERTENCE_AO_DEPT]->(d)
    RETURN p, d
    """
    with driver.session() as session:
        try:
            print("Inserindo relacoes professores-departamentos...")
            result = session.run(query, relacoes=edges_list)
            if verbose:
                for record in result:
                    print(f"Relação PERTENCE_A adicionada entre {record['p']['lattes']} e {record['d']['nome']}.")
        except Exception as e:
            print(f"Erro: {e}")
        finally:
            session.close()
            print("Relacoes profs-deptos inseridas!")

In [21]:
def insert_prod_nodes(driver, prod_list, verbose=False):
    '''Insere nós das produções no banco do Neo4j através do driver.'''
    query = """
    UNWIND $producoes AS prod
    MERGE (n:Producao {
        titulo: prod.titulo, 
        ano: prod.ano, 
        tipo: prod.tipo
    })
    RETURN n
    """
    with driver.session() as session:
        try:
            print("Inserindo producoes...")
            result = session.run(query, producoes=prod_list)
            if verbose:
                for record in result:
                    print(f"Produção adicionada: {record['n']['titulo']} ({record['n']['ano']}).")
        except Exception as e:
            print(f"Erro: {e}")
        finally:
            session.close()
            print("Producoes inseridas!")

In [22]:
def insert_prod_nodes_batch(driver, prod_list, batch_size=1000, verbose=False):
    '''Insere nós das produções em lotes para otimizar a inserção no Neo4j.'''
    query = """
    UNWIND $producoes AS prod
    MERGE (n:Producao {
        titulo: prod.titulo, 
        ano: prod.ano, 
        tipo: prod.tipo
    })
    RETURN n
    """
    with driver.session() as session:
        try:
            print("Inserindo produções em lotes...")
            for i in range(0, len(prod_list), batch_size):
                batch = prod_list[i:i + batch_size]
                result = session.run(query, producoes=batch)
                if verbose:
                    for record in result:
                        print(f"Produção adicionada: {record['n']['titulo']} ({record['n']['ano']}).")
        except Exception as e:
            print(f"Erro: {e}")
        finally:
            print("Inserção de produções finalizada.")

In [23]:

def insert_rel_prod_prof(driver, prod_list, batch_size=1000, verbose=False):
    '''Insere relacao entre professores e producoes no banco.'''
    query = """
    UNWIND $relacoes AS rel
    MATCH (prof:Professor {lattes: rel.lattes_criador})
    MATCH (prod:Producao {titulo: rel.titulo, ano: rel.ano})
    MERGE (prof)-[r:PRODUZ]->(prod)
    RETURN prof, r, prod
    """
    with driver.session() as session:
        try:
            print("Inserindo relacoes producao-professores...")
            for i in range(0, len(prod_list), batch_size):
                batch = prod_list[i:i + batch_size]
                result = session.run(query, relacoes=batch)
                if verbose:
                    for record in result:
                        print(f"Relação PRODUZ adicionada entre {record['prof']['lattes']} e {record['prod']['titulo']}.")
        except Exception as e:
            print(f"Erro: {e}")
        finally:
            session.close()
            print("Relacoes prod-profs inseridas!")

In [24]:
esvazia_banco(driver)
insert_dpt_nodes(driver, deptos_list, verbose=False)
insert_prof_nodes(driver, prof_list, verbose=False)
insert_rel_prof_dept(driver, edges_list, verbose=False)
insert_prod_nodes_batch(driver, prod_list, verbose=False)
insert_rel_prod_prof(driver, prod_list, verbose=False)

Banco esvaziado.
Inserindo departamentos...
Departamentos inseridos!
Inserindo professores...
Professores inseridos!
Inserindo relacoes professores-departamentos...
Relacoes profs-deptos inseridas!
Inserindo produções em lotes...
Inserção de produções finalizada.
Inserindo relacoes producao-professores...
Relacoes prod-profs inseridas!
